# 03 — Bayesian Fair-Value Model

Builds a feature table (momentum, macro signals, time-to-resolution,
liquidity), then fits two fair-value estimators:

1. **Bayesian logit-update model** — interpretable, prior = yesterday's
   market probability, evidence = standardized macro surprises, updated
   additively in logit space.
2. **Gradient boosted regressor** — flexible non-linear benchmark.

Then compares each model's fair value to the market price to surface
mispricings.


In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from feature_engineering import build_feature_table, FEATURE_COLUMNS
from fair_value_model import BayesianUpdateModel, GradientBoostFairValueModel, time_based_train_test_split
import visualization as viz

pd.set_option("display.max_columns", 25)
%matplotlib inline

In [ ]:
markets_df = pd.read_csv("../data/raw/markets.csv")
macro_df = pd.read_csv("../data/raw/macro_features.csv", parse_dates=["date"])
clean_df = pd.read_csv("../data/processed/clean_probabilities.csv", parse_dates=["timestamp"])

features_df = build_feature_table(clean_df, markets_df, macro_df)
print(features_df.shape)
features_df[["market_id", "outcome_name", "timestamp"] + FEATURE_COLUMNS + ["target_prob"]].head()

## Time-based train/test split

Never train on the future — split chronologically, not randomly.

In [ ]:
train_df, test_df = time_based_train_test_split(features_df, test_frac=0.25)
print(f"train: {train_df.shape}, test: {test_df.shape}")
print(f"train range: {train_df['timestamp'].min()} -> {train_df['timestamp'].max()}")
print(f"test range:  {test_df['timestamp'].min()} -> {test_df['timestamp'].max()}")

## Fit the Bayesian fair-value model

In [ ]:
bayes_model = BayesianUpdateModel().fit(train_df)

print("Logit-space coefficients (per 1 std-dev of evidence):")
bayes_model.coefficients()

In [ ]:
test_df = test_df.copy()
test_df["fair_prob"] = bayes_model.predict(test_df)
test_df["edge"] = test_df["fair_prob"] - test_df["clean_prob"]

test_df[["market_id", "outcome_name", "timestamp", "clean_prob", "fair_prob", "edge"]].sort_values(
    "edge", key=abs, ascending=False
).head(15)

## Benchmark: gradient boosted regressor

In [ ]:
gbr_model = GradientBoostFairValueModel().fit(train_df)
test_df["fair_prob_gbr"] = gbr_model.predict(test_df)

print("Feature importance:")
gbr_model.feature_importance()

## Largest current mispricings (Bayesian model)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
viz.plot_mispricing_table(test_df, top_n=15, ax=ax)
plt.tight_layout()
plt.show()

## Calibration check

If the model is well-calibrated, points should sit near the diagonal:
predicted probability ≈ realized frequency.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
viz.plot_calibration(test_df, ax=ax)
plt.show()

## Persist scored test set for the backtest notebook

In [ ]:
test_df.to_csv("../data/processed/scored_test_set.csv", index=False)
print(f"Saved {test_df.shape[0]} scored rows to data/processed/scored_test_set.csv")